# Fish detection on OzFish (reef fish) - pretrained CFD RF-DETR Medium vs our detectors - Colab, ~30 min

**OzFish** (AIMS / UWA / Curtin, CC-BY 3.0 AU, doi 10.25845/5e28f062c5097): 1,758 baited-camera reef frames,
43,572 human-drawn fish boxes, 910 camera drops. **Not** among the 17 datasets the Community Fish Detector was
trained on, and never used to train our detectors, so it is a clean test for all models.

- Split **by camera drop** (left and right stereo views of a drop stay together): 30% val -> picks each model's
  confidence threshold (max F1); 70% test -> reported once.
- Same matching rule as all project evaluations: greedy, highest confidence first, IoU >= 0.5. Also AP50.
  95% intervals: bootstrap over test camera drops.
- Models: CFD RF-DETR Medium (pretrained, no training) and, if found in `MyDrive/fish/`, our `best_fish.pt` and
  `best_model.pt` (16-class, fish class) for comparison.

**Runtime:** T4 GPU, then Run all. Frames (~3.2 GB) are downloaded from the public Pawsey portal inside Colab.
**Bring back:** `MyDrive/fish/ozfish_results.json`.

In [ ]:
!nvidia-smi -L
!pip install -q rfdetr supervision ultralytics
!wget -q -nc -O /content/cfd_rfdetr_medium.pth https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-medium-1024-2026.03.24.cp-011.20260706-release.pth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, hashlib, collections, requests
import numpy as np
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
DRIVE_DIR = '/content/drive/MyDrive/fish'; os.makedirs(DRIVE_DIR, exist_ok=True)
OZ = Path('/content/ozfish'); (OZ / 'frames').mkdir(parents=True, exist_ok=True)
API = 'https://storage.pawsey.org.au/api'
TOKEN = requests.post(f'{API}/login/public', timeout=60).json()['token']   # public, no login

def urls_for(paths):
    r = requests.post(f'{API}/m/public/download/prepare', headers={'X-Auth-Token': TOKEN},
                      json={'filePaths': paths}, timeout=120).json()
    assert r.get('success'), r
    return {x['filePath']: x['url'] for x in r['result']}

def fetch(url, dest, tries=4):
    for t in range(tries):
        try:
            data = requests.get(url, timeout=180).content
            if len(data) > 1000:
                dest.write_bytes(data); return True
        except Exception:
            pass
    return False

# manifests (bounding boxes)
man = [f'/FDFML/labelled/manifests/batch0{i}.json' for i in (1, 2, 3)]
for p, u in urls_for(man).items():
    fetch(u, OZ / Path(p).name)
RECS = []
for i in (1, 2, 3):
    for line in (OZ / f'batch0{i}.json').read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            ann = next(v for k, v in r.items() if isinstance(v, dict) and 'annotations' in v)
            RECS.append((r['source-ref'], ann['annotations']))
print(len(RECS), 'frames,', sum(len(a) for _, a in RECS), 'boxes')

In [ ]:
need = [n for n, _ in RECS if not (OZ / 'frames' / n).exists()]
print('downloading', len(need), 'frames')
ok = 0
for i in range(0, len(need), 50):
    chunk = need[i:i + 50]
    urls = urls_for([f'/FDFML/frames/{n}' for n in chunk])
    with ThreadPoolExecutor(16) as ex:
        ok += sum(ex.map(lambda n: fetch(urls[f'/FDFML/frames/{n}'], OZ / 'frames' / n), chunk))
    print(i + len(chunk), '/', len(need))
RECS = [(n, a) for n, a in RECS if (OZ / 'frames' / n).exists()]
print('frames available:', len(RECS))

In [ ]:
drop = lambda n: re.match(r'([A-Za-z]+\d+)', n).group(1)          # camera drop, L/R views together
split = lambda n: 'val' if int(hashlib.md5(drop(n).encode()).hexdigest(), 16) % 10 < 3 else 'test'
GT = {n: np.array([[a['left'], a['top'], a['left'] + a['width'], a['top'] + a['height']] for a in ann], np.float32).reshape(-1, 4)
      for n, ann in RECS}
SPLITS = {s: sorted(n for n in GT if split(n) == s) for s in ('val', 'test')}
for s, ns in SPLITS.items():
    print(s, len(ns), 'frames', len({drop(n) for n in ns}), 'drops', sum(len(GT[n]) for n in ns), 'fish')

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match_flags(gt, boxes, confs):
    order = np.argsort(-confs)
    ious = iou(boxes, gt) if len(boxes) and len(gt) else np.zeros((len(boxes), len(gt)))
    used, out = set(), []
    for i in order:
        cand = [(ious[i, g], g) for g in range(len(gt)) if g not in used and ious[i, g] >= 0.5]
        if cand:
            used.add(max(cand)[1])
        out.append((float(confs[i]), bool(cand)))
    return out

def counts(gt, boxes, confs, conf):
    k = confs >= conf
    f = match_flags(gt, boxes[k], confs[k]); tp = sum(x for _, x in f)
    return tp, len(f) - tp, len(gt) - tp

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def evaluate(name, preds):
    tot = lambda ns, c: tuple(sum(v) for v in zip(*(counts(GT[n], *preds[n], c) for n in ns)))
    table = [(prf(*tot(SPLITS['val'], c))[2], float(c)) for c in np.round(np.arange(0.05, 0.91, 0.05), 2)]
    vf1, conf = max(table)
    ns = SPLITS['test']
    per = {n: counts(GT[n], *preds[n], conf) for n in ns}
    tp, fp, fn = (sum(v[k] for v in per.values()) for k in range(3)); p, r, f1 = prf(tp, fp, fn)
    groups = collections.defaultdict(list)
    for n in ns:
        groups[drop(n)].append(n)
    rng = np.random.default_rng(0); keys = list(groups); boot = []
    for _ in range(1000):
        s = [n for k in rng.choice(keys, len(keys)) for n in groups[k]]
        boot.append(prf(*(sum(per[n][k] for n in s) for k in range(3))))
    ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]
    flags = sorted((f for n in ns for f in match_flags(GT[n], *preds[n])), key=lambda t: -t[0])
    n_gt = sum(len(GT[n]) for n in ns)
    tps = np.cumsum([f for _, f in flags]); fps = np.cumsum([not f for _, f in flags])
    mrec = np.concatenate([[0], tps / n_gt, [1]]); mpre = np.concatenate([[1], tps / np.maximum(tps + fps, 1), [0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    ap50 = float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))
    res = {'conf_chosen_on_val': conf, 'val_f1': round(vf1, 3), 'precision': round(p, 3), 'recall': round(r, 3),
           'f1': round(f1, 3), 'ap50': round(ap50, 3), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn),
           'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2)}
    print(f'{name:28s} conf {conf:.2f} | TEST P {p:.3f} R {r:.3f} F1 {f1:.3f} {ci(2)} AP50 {ap50:.3f}')
    return res

ALL = SPLITS['val'] + SPLITS['test']
RESULTS = {'dataset': 'OzFish bounding boxes (1,758 frames, 43,572 boxes); split by camera drop 30% val / 70% test',
           'test_frames': len(SPLITS['test']), 'test_drops': len({drop(n) for n in SPLITS['test']}),
           'test_fish': int(sum(len(GT[n]) for n in SPLITS['test'])), 'matching': 'greedy IoU>=0.5', 'models': {}}

In [ ]:
# 1) Pretrained Community Fish Detector RF-DETR Medium (no training)
from rfdetr import from_checkpoint
cfd = from_checkpoint('/content/cfd_rfdetr_medium.pth')
P = {}
for i, n in enumerate(ALL):
    d = cfd.predict(Image.open(OZ / 'frames' / n).convert('RGB'), threshold=0.02)
    P[n] = (np.asarray(d.xyxy, np.float32).reshape(-1, 4), np.asarray(d.confidence, np.float32).reshape(-1))
    if i % 300 == 0: print(i, '/', len(ALL))
RESULTS['models']['CFD RF-DETR Medium (pretrained)'] = evaluate('CFD RF-DETR Medium', P)
del cfd

In [ ]:
# 2) Our detectors, if uploaded to MyDrive/fish/ (best_fish.pt: fish-only; best_model.pt: 16-class TrashCan, class 2 = fish)
from ultralytics import YOLO
for fname, cls, label in (('best_fish.pt', 0, 'Our fish detector (best_fish.pt)'),
                          ('best_model.pt', 2, 'Our 16-class detector (fish class)')):
    path = Path(DRIVE_DIR) / fname
    if not path.exists():
        print('skip', fname, '(not in', DRIVE_DIR, ')'); continue
    m = YOLO(str(path)); P = {}
    for i in range(0, len(ALL), 16):
        names = ALL[i:i + 16]
        for n, r in zip(names, m.predict([str(OZ / 'frames' / x) for x in names], conf=0.02, imgsz=640, device=0, verbose=False)):
            k = r.boxes.cls.cpu().numpy() == cls
            P[n] = (r.boxes.xyxy.cpu().numpy()[k].astype(np.float32), r.boxes.conf.cpu().numpy()[k].astype(np.float32))
    RESULTS['models'][label] = evaluate(label, P)

json.dump(RESULTS, open(f'{DRIVE_DIR}/ozfish_results.json', 'w'), indent=2)
print(json.dumps(RESULTS, indent=1))